In [2]:
import numpy as np
import pandas as pd
import yaml
from pathlib import Path

root = Path("/Users/gherardi/Documents/GitHub/glider_optimization")
cfg_path = root / "conf/test.yaml"
cfg = yaml.safe_load(cfg_path.read_text()) or {}

plane = cfg.get("plane", {})
dyn = plane.get("dyn", {})
wing = plane.get("wing", {})
elevator = plane.get("elevator", {})
nf = cfg.get("neuralFoilSampling", {})

# === Constants mirrored from glider_jinenv.initDyn(), now read from config ===
m = float(dyn.get("mass", 0.065))
l_w_i = float(dyn.get("l_w_i", -0.005))
l_w_f = float(dyn.get("l_w_f", -0.015))
l = float(dyn.get("l", 0.26))
l_3d = float(dyn.get("l_3d", 0.344))  # wing LE -> elevator LE (3D anchor distance)
l_e = float(dyn.get("l_e", 0.02))
S_w = float(dyn.get("S_w", 0.158))
S_e = float(dyn.get("S_e", 0.017))

# Same helper as glider_jinenv.mc_to_wcom(l_w): return l_w + 0.003
def mc_to_wcom(l_w):
    return l_w + 0.003

# Lightweight local fallback (if aero_rom helper import fails)
def spanwise_centroid_x_fallback(y_half, c_half, xle_half):
    y = np.asarray(y_half, dtype=float)
    c = np.asarray(c_half, dtype=float)
    xle = np.asarray(xle_half, dtype=float)
    c_mid = 0.5 * (c[:-1] + c[1:])
    xle_mid = 0.5 * (xle[:-1] + xle[1:])
    dy = np.abs(y[1:] - y[:-1])
    w = (c_mid ** 2) * dy
    x_mid = xle_mid + 0.4 * c_mid
    return float(np.sum(x_mid * w) / np.sum(w))

def spanwise_centroid_z_fallback(y_half, c_half, dihedral_deg):
    y = np.asarray(y_half, dtype=float)
    c = np.asarray(c_half, dtype=float)
    y_mid = 0.5 * (y[:-1] + y[1:])
    c_mid = 0.5 * (c[:-1] + c[1:])
    dy = np.abs(y[1:] - y[:-1])
    z_line = np.tan(np.deg2rad(float(dihedral_deg))) * np.abs(y_mid)
    w = (c_mid ** 2) * dy
    return float(np.sum(z_line * w) / np.sum(w))

# Prefer exact helper logic used to generate 3D checkpoint centroids
have_exact_helpers = False
try:
    from aero_rom.src.diff_pipeline import _spanwise_centroid_x, _spanwise_centroid_z
    have_exact_helpers = True
except Exception:
    pass

# Shared derived values
m_w = 0.6 * m * S_w / (S_w + S_e)
m_e = 0.6 * m * S_e / (S_w + S_e)
m_f = 0.4 * m
l_w = 0.5 * (l_w_i + l_w_f)
l_w_m = (l_w_i + l_w_f) / 2
com_w_2d = l_w_m + mc_to_wcom(l_w_m)
com_e_2d = l + l_e
l_f = -(l_w * m_w + (l + l_e) * m_e) / m_f
com_f = l_f

# 2D aerodynamic reference centroid (scalar x)
com_a_2d = (com_w_2d * m_w + com_e_2d * m_e + com_f * m_f) / (m_w + m_e + m_f)

# 3D centroids from geometry in conf/test.yaml
wing_airfoil = wing.get("airfoil_root", wing.get("airfoil", "NACA4412"))
elev_airfoil = elevator.get("airfoil", "NACA0012")

if have_exact_helpers:
    wing_x_local = _spanwise_centroid_x(wing["y_half"], wing["c_half"], wing["xle_half"], wing_airfoil)
    wing_z_local = _spanwise_centroid_z(wing["y_half"], wing["c_half"], wing.get("dihedral", 0.0), wing_airfoil)
    elev_x_local = _spanwise_centroid_x(elevator["y_half"], elevator["c_half"], elevator["xle_half"], elev_airfoil)
    elev_z_local = _spanwise_centroid_z(elevator["y_half"], elevator["c_half"], elevator.get("dihedral", 0.0), elev_airfoil)
    centroid_src = "geometry (exact helper from aero_rom.src.diff_pipeline)"
else:
    wing_x_local = spanwise_centroid_x_fallback(wing["y_half"], wing["c_half"], wing["xle_half"])
    wing_z_local = spanwise_centroid_z_fallback(wing["y_half"], wing["c_half"], wing.get("dihedral", 0.0))
    elev_x_local = spanwise_centroid_x_fallback(elevator["y_half"], elevator["c_half"], elevator["xle_half"])
    elev_z_local = spanwise_centroid_z_fallback(elevator["y_half"], elevator["c_half"], elevator.get("dihedral", 0.0))
    centroid_src = "geometry (fallback approximation; aero_rom helper unavailable)"

# Match glider_jinenv 3D anchor convention:
# - wing LE at l_w_i
# - elevator LE at l_w_i + l_3d
x_w_le = l_w_i
x_e_le = l_w_i + l_3d

# Map local geometry centroids to body frame used in glider_jinenv
com_w_x_raw = float(x_w_le + wing_x_local)
com_w_z_raw = float(wing_z_local)
com_e_x_raw = float(x_e_le + elev_x_local)
com_e_z_raw = float(elev_z_local)

# Recenter so full plane centroid is at (0, 0) in 3D mode
com_a_x_raw = (com_w_x_raw * m_w + com_e_x_raw * m_e + com_f * m_f) / (m_w + m_e + m_f)
com_a_z_raw = (com_w_z_raw * m_w + com_e_z_raw * m_e + 0.0 * m_f) / (m_w + m_e + m_f)

com_w_x = com_w_x_raw - com_a_x_raw
com_w_z = com_w_z_raw - com_a_z_raw
com_e_x = com_e_x_raw - com_a_x_raw
com_e_z = com_e_z_raw - com_a_z_raw
com_a_x_3d = 0.0
com_a_z_3d = 0.0

# Body-frame lever arms used later in 3D torque path
r_w_bx_3d = -com_w_x + com_a_x_3d
r_w_bz_3d = -com_w_z + com_a_z_3d
r_e_bx_3d = -com_e_x + com_a_x_3d
r_e_bz_3d = -com_e_z + com_a_z_3d

# 2D equivalent lever scalar (before world rotation)
r_w_2d_scalar = -com_w_2d + com_a_2d
r_e_2d_scalar = -com_e_2d + com_a_2d

print("Centroid source for 3D:", centroid_src)
print("use_3d_llt:", bool(nf.get("use_3d_llt", False)))
print("3D anchor convention: x_w_le=l_w_i, x_e_le=l_w_i+l_3d")
print("x_w_le, x_e_le:", x_w_le, x_e_le)
print("Plane centroid after recenter (expected ~0,0):", com_a_x_3d, com_a_z_3d)

summary = pd.DataFrame([
    {
        "mode": "2D",
        "com_w_x": com_w_2d, "com_w_z": 0.0,
        "com_e_x": com_e_2d, "com_e_z": 0.0,
        "com_a_x": com_a_2d, "com_a_z": 0.0,
    },
    {
        "mode": "3D_raw_from_geometry",
        "com_w_x": com_w_x_raw, "com_w_z": com_w_z_raw,
        "com_e_x": com_e_x_raw, "com_e_z": com_e_z_raw,
        "com_a_x": com_a_x_raw, "com_a_z": com_a_z_raw,
    },
    {
        "mode": "3D_recentered",
        "com_w_x": com_w_x, "com_w_z": com_w_z,
        "com_e_x": com_e_x, "com_e_z": com_e_z,
        "com_a_x": com_a_x_3d, "com_a_z": com_a_z_3d,
    },
])
display(summary)

lever_cmp = pd.DataFrame([
    {
        "mode": "2D",
        "r_w_bx_or_scalar": r_w_2d_scalar, "r_w_bz": 0.0,
        "r_e_bx_or_scalar": r_e_2d_scalar, "r_e_bz": 0.0,
    },
    {
        "mode": "3D_recentered",
        "r_w_bx_or_scalar": r_w_bx_3d, "r_w_bz": r_w_bz_3d,
        "r_e_bx_or_scalar": r_e_bx_3d, "r_e_bz": r_e_bz_3d,
    },
])
display(lever_cmp)

delta = summary.iloc[2][["com_w_x", "com_w_z", "com_e_x", "com_e_z", "com_a_x", "com_a_z"]] - \
        summary.iloc[0][["com_w_x", "com_w_z", "com_e_x", "com_e_z", "com_a_x", "com_a_z"]]
print("3D_recentered - 2D centroid deltas:")
display(delta.to_frame("delta").T)

Centroid source for 3D: geometry (exact helper from aero_rom.src.diff_pipeline)
use_3d_llt: True
3D anchor convention: x_w_le=l_w_i, x_e_le=l_w_i+l_3d
x_w_le, x_e_le: -0.005 0.33899999999999997
Plane centroid after recenter (expected ~0,0): 0.0 0.0


,mode,com_w_x,com_w_z,com_e_x,com_e_z,com_a_x,com_a_z
0,2D,-0.017000,0.000000,0.280000,0.000000e+00,-0.003792,0.000000
1,3D_raw_from_geometry,0.092125,0.020381,0.364227,2.110199e-19,0.060232,0.011041
2,3D_recentered,0.031893,0.009340,0.303995,-1.104070e-02,0.000000,0.000000


,mode,r_w_bx_or_scalar,r_w_bz,r_e_bx_or_scalar,r_e_bz
0,2D,0.013208,0.00000,-0.283792,0.000000
1,3D_recentered,-0.031893,-0.00934,-0.303995,0.011041


3D_recentered - 2D centroid deltas:


,com_w_x,com_w_z,com_e_x,com_e_z,com_a_x,com_a_z
delta,0.048893,0.00934,0.023995,-0.011041,0.003792,0.0


In [2]:
from pathlib import Path
import pprint
import torch

pt_path = Path("/Users/gherardi/Documents/GitHub/glider_optimization_debug/artifacts/models/3d_blocks.pt")
print("Path:", pt_path)
print("Exists:", pt_path.exists())

obj = torch.load(str(pt_path), map_location="cpu")
print("Top-level type:", type(obj))

if isinstance(obj, dict):
    print("\nTop-level keys:")
    for k in sorted(obj.keys()):
        v = obj[k]
        shape = tuple(v.shape) if hasattr(v, "shape") else None
        print(f"- {k}: type={type(v).__name__}, shape={shape}")

    # Print centroid/details if present
    for special in ["centroid", "meta", "config", "state_dict"]:
        if special in obj:
            print(f"\n{special}:")
            if isinstance(obj[special], dict):
                pprint.pprint({kk: obj[special][kk] for kk in list(obj[special].keys())[:30]})
            else:
                print(obj[special])
else:
    print("Object preview:")
    print(obj)

Path: /Users/gherardi/Documents/GitHub/glider_optimization_debug/artifacts/models/3d_blocks.pt
Exists: True
Top-level type: <class 'dict'>

Top-level keys:
- beta: type=float, shape=None
- centroid: type=dict, shape=None
- config_path: type=str, shape=None
- device: type=str, shape=None
- elevator_geometry: type=dict, shape=None
- elevator_requires_grad: type=bool, shape=None
- enforce_symmetry: type=bool, shape=None
- flow: type=dict, shape=None
- model_size: type=str, shape=None
- n_iter: type=int, shape=None
- tol: type=float, shape=None
- wing_geometry: type=dict, shape=None
- wing_requires_grad: type=bool, shape=None

centroid:
{'elevator_x': 0.02522654693991627,
 'elevator_z': 2.1101987955775332e-19,
 'wing_x': 0.09712500253101201,
 'wing_z': 0.02038103043897718}


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

root = Path("/Users/gherardi/Documents/GitHub/glider_optimization_debug/diagnostics/2026-02-13_3d-llt-debug")
f2 = root / "rollout_terms_stage_0_16_full_test2a_3dwing_fresh.csv"
f3 = root / "rollout_terms_stage_0_16_full_test5_3dwing_3delev_centroidLplusX_20260219_170659.csv"

def check(file):
    df = pd.read_csv(file).copy()
    # two candidate conventions
    df["tau_from_plus"] = df["r_e_x"]*df["F_ez"] - df["r_e_z"]*df["F_ex"] + df["M_e"]
    df["tau_from_minus"] = -df["r_e_x"]*df["F_ez"] + df["r_e_z"]*df["F_ex"] + df["M_e"]
    df["err_plus"] = df["tau_e"] - df["tau_from_plus"]
    df["err_minus"] = df["tau_e"] - df["tau_from_minus"]

    print("\nFILE:", file.name)
    print("mean |tau_e - tau_from_plus| :", np.nanmean(np.abs(df["err_plus"])))
    print("mean |tau_e - tau_from_minus|:", np.nanmean(np.abs(df["err_minus"])))
    display(df.loc[:3, [
        "stage","r_e_x","r_e_z","F_ex","F_ez","M_e",
        "tau_e_term_rxfz","tau_e_term_rzfx","tau_e",
        "tau_from_plus","tau_from_minus"
    ]])

check(f2)
check(f3)


FILE: rollout_terms_stage_0_16_full_test2a_3dwing_fresh.csv
mean |tau_e - tau_from_plus| : 0.050334770708270436
mean |tau_e - tau_from_minus|: 9.122604632489797e-17


,stage,r_e_x,r_e_z,F_ex,F_ez,M_e,tau_e_term_rxfz,tau_e_term_rzfx,tau_e,tau_from_plus,tau_from_minus
0,0,-0.281461,-0.000000,2.793966e-08,-0.419095,0.000937,-0.117959,-0.000000,-0.117022,0.118896,-0.117022
1,1,-0.281461,-0.000000,1.035532e-08,-0.231912,0.000556,-0.065274,-0.000000,-0.064718,0.065831,-0.064718
2,2,-0.281407,-0.005468,2.263800e-03,-0.116506,0.000288,-0.032786,-0.000012,-0.032510,0.033086,-0.032510
3,3,-0.281137,-0.013496,1.869643e-03,-0.038948,0.000097,-0.010950,-0.000025,-0.010878,0.011072,-0.010878



FILE: rollout_terms_stage_0_16_full_test5_3dwing_3delev_centroidLplusX_20260219_170659.csv
mean |tau_e - tau_from_plus| : 5.953584315071298e+238
mean |tau_e - tau_from_minus|: 1.1652569176435556e+254


,stage,r_e_x,r_e_z,F_ex,F_ez,M_e,tau_e_term_rxfz,tau_e_term_rzfx,tau_e,tau_from_plus,tau_from_minus
0,0,-0.224559,-2.110199e-19,0.046321,-0.442899,0.001042,0.099457,9.774662e-21,0.100499,0.100499,-0.098415
1,1,-0.224559,-2.110199e-19,0.001246,-0.726024,0.001447,0.163036,2.629258e-22,0.164482,0.164482,-0.161589
2,2,-0.223927,1.683921e-02,-0.105938,-1.408763,0.002243,0.315460,1.783910e-03,0.319487,0.319487,-0.315001
3,3,-0.217780,5.475958e-02,-0.822619,-3.271579,0.003602,0.712486,4.504626e-02,0.761134,0.761134,-0.753930


In [5]:
import numpy as np
import yaml
from pathlib import Path
from math import sqrt

root = Path("/Users/gherardi/Documents/GitHub/glider_optimization")
cfg_path = root / "conf/test.yaml"
cfg = yaml.safe_load(cfg_path.read_text()) or {}
nf = cfg.get("neuralFoilSampling", {})

def chebyshev_nodes(a, b, n):
    k = np.arange(n)
    return 0.5 * (a + b) + 0.5 * (b - a) * np.cos((2 * k + 1) / (2 * n) * np.pi)

n_samples = int(nf.get("n_samples", 100))
n_1d = int(sqrt(n_samples))

AoA_min = float(nf.get("AoA_min", -10.0))
AoA_max = float(nf.get("AoA_max", 20.0))
Re_min = float(nf.get("Re_min", 2e4))
Re_max = float(nf.get("Re_max", 2e5))

aoa_1d = chebyshev_nodes(AoA_min, AoA_max, n_1d)
re_1d = chebyshev_nodes(Re_min, Re_max, n_1d)

aoa_grid, re_grid = np.meshgrid(aoa_1d, re_1d, indexing="ij")
alpha_batch = aoa_grid.reshape(-1)
Re_batch = re_grid.reshape(-1)

# Ascending-order views for readability
aoa_1d_asc = np.sort(aoa_1d)
re_1d_asc = np.sort(re_1d)
alpha_batch_asc = np.sort(alpha_batch)
Re_batch_asc = np.sort(Re_batch)

print(f"n_samples={n_samples}, n_1d={n_1d}, total_grid_points={alpha_batch.size}")
print(f"AoA range: [{alpha_batch.min():.6f}, {alpha_batch.max():.6f}] deg")
print(f"Re range : [{Re_batch.min():.6f}, {Re_batch.max():.6f}]")

print("\nAoA 1D nodes (deg) - ascending:")
print(aoa_1d_asc.tolist())
print(f"AoA 1D nodes - length: {len(aoa_1d_asc)}")

print("\nRe 1D nodes - ascending:")
print(re_1d_asc.tolist())
print(f"Re 1D nodes - length: {len(re_1d_asc)}")

print("\nFlattened alpha_batch (deg) - ascending:")
print(alpha_batch_asc.tolist())
print(f"Flattened alpha_batch - length: {len(alpha_batch_asc)}")

print("\nFlattened Re_batch - ascending:")
print(Re_batch_asc.tolist())
print(f"Flattened Re_batch - length: {len(Re_batch_asc)}")

n_samples=1024, n_1d=32, total_grid_points=1024
AoA range: [-29.963864, 29.963864] deg
Re range : [160.166963, 99939.833037]

AoA 1D nodes (deg) - ascending:
[-29.96386368615517, -29.675295298943432, -29.10093759583632, -28.24632195549062, -27.1196787937033, -25.73185830000816, -24.096225944419345, -22.228533760648766, -20.146768645410553, -17.870979134773, -15.423082325796646, -12.826652802908455, -10.106695601766598, -7.289405397097916, -4.401914233660849, -1.4720302298225403, 1.472030229822544, 4.401914233660852, 7.289405397097919, 10.106695601766601, 12.826652802908466, 15.423082325796653, 17.870979134773005, 20.146768645410553, 22.228533760648777, 24.09622594441935, 25.731858300008163, 27.1196787937033, 28.246321955490625, 29.10093759583632, 29.675295298943432, 29.96386368615517]
AoA 1D nodes - length: 32

Re 1D nodes - ascending:
[160.16696255163697, 640.6333272591874, 1596.938902932532, 3019.8739441081198, 4895.734808484005, 7206.455930486416, 9929.783802541788, 13039.4912885198

In [7]:
import re
import numpy as np
import yaml
from pathlib import Path
from math import sqrt

# =========================
# User inputs
# =========================
airfoil_name = "Optimised"  # Change this to any foil name you want
template_xml = Path("/Users/gherardi/Documents/GitHub/glider_optimization/artifacts/xml/T1_Re0_267_M0_00_N9_0.xml")

# =========================
# Build 2D Reynolds grid exactly like NeuralFoilSampling
# =========================
root = Path("/Users/gherardi/Documents/GitHub/glider_optimization")
cfg_path = root / "conf/test.yaml"
cfg = yaml.safe_load(cfg_path.read_text()) or {}
nf = cfg.get("neuralFoilSampling", {})

def chebyshev_nodes(a, b, n):
    k = np.arange(n)
    return 0.5 * (a + b) + 0.5 * (b - a) * np.cos((2 * k + 1) / (2 * n) * np.pi)

n_samples = int(nf.get("n_samples", 100))
n_1d = int(sqrt(n_samples))
Re_min = float(nf.get("Re_min", 2e4))
Re_max = float(nf.get("Re_max", 2e5))

re_1d = chebyshev_nodes(Re_min, Re_max, n_1d)
re_vals = np.sort(np.unique(np.rint(re_1d).astype(int)))  # rounded integer Reynolds values

print(f"Template: {template_xml}")
print(f"Output dir: {template_xml.parent}")
print(f"Airfoil: {airfoil_name}")
print(f"Reynolds values ({len(re_vals)}): {re_vals.tolist()}")

# =========================
# Create one XML per Reynolds number
# =========================
xml_template_text = template_xml.read_text(encoding="utf-8")
out_dir = template_xml.parent
written_files = []

for re_val in re_vals:
    # Naming convention requested by user
    polar_name = f"T1_Re0.{re_val}_M0.00_N9.0"
    out_name = f"T1_Re0_{re_val}_M0_00_N9_0.xml"
    out_path = out_dir / out_name

    txt = xml_template_text

    # Replace Polar_Name / Foil_Name / Fixed_Reynolds values
    txt = re.sub(r"<Polar_Name>.*?</Polar_Name>", f"<Polar_Name>{polar_name}</Polar_Name>", txt, flags=re.DOTALL)
    txt = re.sub(r"<Foil_Name>.*?</Foil_Name>", f"<Foil_Name>{airfoil_name}</Foil_Name>", txt, flags=re.DOTALL)
    txt = re.sub(r"<Fixed_Reynolds>.*?</Fixed_Reynolds>", f"<Fixed_Reynolds>{re_val}</Fixed_Reynolds>", txt, flags=re.DOTALL)

    # Write (overwrite if exists)
    out_path.write_text(txt, encoding="utf-8")
    written_files.append(out_path.name)

print(f"\nDone. Wrote/overwrote {len(written_files)} XML files:")
for f in written_files:
    print(f"- {f}")

Template: /Users/gherardi/Documents/GitHub/glider_optimization/artifacts/xml/T1_Re0_267_M0_00_N9_0.xml
Output dir: /Users/gherardi/Documents/GitHub/glider_optimization/artifacts/xml
Airfoil: Optimised
Reynolds values (32): [160, 641, 1597, 3020, 4896, 7206, 9930, 13039, 16506, 20295, 24371, 28694, 33222, 37913, 42721, 47599, 52501, 57379, 62187, 66878, 71406, 75729, 79805, 83594, 87061, 90170, 92894, 95204, 97080, 98503, 99459, 99940]

Done. Wrote/overwrote 32 XML files:
- T1_Re0_160_M0_00_N9_0.xml
- T1_Re0_641_M0_00_N9_0.xml
- T1_Re0_1597_M0_00_N9_0.xml
- T1_Re0_3020_M0_00_N9_0.xml
- T1_Re0_4896_M0_00_N9_0.xml
- T1_Re0_7206_M0_00_N9_0.xml
- T1_Re0_9930_M0_00_N9_0.xml
- T1_Re0_13039_M0_00_N9_0.xml
- T1_Re0_16506_M0_00_N9_0.xml
- T1_Re0_20295_M0_00_N9_0.xml
- T1_Re0_24371_M0_00_N9_0.xml
- T1_Re0_28694_M0_00_N9_0.xml
- T1_Re0_33222_M0_00_N9_0.xml
- T1_Re0_37913_M0_00_N9_0.xml
- T1_Re0_42721_M0_00_N9_0.xml
- T1_Re0_47599_M0_00_N9_0.xml
- T1_Re0_52501_M0_00_N9_0.xml
- T1_Re0_57379_M0_00_N9

In [8]:
import numpy as np
import pandas as pd
import yaml
from pathlib import Path
from math import sqrt

# Build the same 2D Reynolds grid (integer-only), then export as one-column CSV for Excel
root = Path("/Users/gherardi/Documents/GitHub/glider_optimization")
cfg_path = root / "conf/test.yaml"
cfg = yaml.safe_load(cfg_path.read_text()) or {}
nf = cfg.get("neuralFoilSampling", {})

def chebyshev_nodes(a, b, n):
    k = np.arange(n)
    return 0.5 * (a + b) + 0.5 * (b - a) * np.cos((2 * k + 1) / (2 * n) * np.pi)

n_samples = int(nf.get("n_samples", 100))
n_1d = int(sqrt(n_samples))
Re_min = float(nf.get("Re_min", 2e4))
Re_max = float(nf.get("Re_max", 2e5))

re_1d = chebyshev_nodes(Re_min, Re_max, n_1d)
re_vals_int = np.sort(np.unique(np.rint(re_1d).astype(int)))

# One-column CSV output for Excel
csv_path = root / "artifacts" / "xml" / "reynolds_2d_grid_int.csv"
csv_path.parent.mkdir(parents=True, exist_ok=True)

pd.DataFrame({"Reynolds": re_vals_int}).to_csv(csv_path, index=False)

print(f"Saved: {csv_path}")
print(f"Count: {len(re_vals_int)}")
print("Values:")
print(re_vals_int.tolist())

Saved: /Users/gherardi/Documents/GitHub/glider_optimization/artifacts/xml/reynolds_2d_grid_int.csv
Count: 32
Values:
[160, 641, 1597, 3020, 4896, 7206, 9930, 13039, 16506, 20295, 24371, 28694, 33222, 37913, 42721, 47599, 52501, 57379, 62187, 66878, 71406, 75729, 79805, 83594, 87061, 90170, 92894, 95204, 97080, 98503, 99459, 99940]
